In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 --no-cache-dir
!pip install -r requirements.txt

In [ ]:
# 주피터 노트북 웹 UI 상에 로그인 위젯 띄우기
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# PyTorch 라이브러리 로드 및 GPU 정보 검증 코드
import torch
import sys

print("Python Version:", sys.version)
print("PyTorch Version:", torch.__version__)

# CUDA 사용 가능 여부 확인
cuda_available = torch.cuda.is_available()
print("CUDA Available:", cuda_available)

if cuda_available:
    # 현재 사용 중인 GPU 디바이스 번호 및 이름 확인
    current_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(current_device)
    # GPU 전체 메모리 크기 측정
    total_memory = torch.cuda.get_device_properties(current_device).total_memory / (1024 ** 3)
    
    print(f"Active GPU ID: {current_device}")
    print(f"Active GPU Name: {device_name}")
    print(f"Total VRAM: {total_memory:.2f} GB")
    device = "cuda"
else:
    print("WARNING: CUDA is not available. Running on CPU mode.")
    device = "cpu"

In [ ]:
# Transformers 라이브러리 및 Auto클래스 임포트
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct"

# CPU/GPU 사양에 대응한 데이터 타입 및 디바이스 매핑 결정
if torch.cuda.is_available():
    torch_dtype = torch.bfloat16
    device_map = "auto"
    low_cpu_mem_usage = True
else:
    torch_dtype = torch.float32
    device_map = "cpu"
    low_cpu_mem_usage = True

print(f"Loading Model... Target Device: {device}, Precision: {torch_dtype}")

# 1) 토크나이저 로드 (공식 가이드 규격 로드)
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

# 2) 모델 로드 (EXAONE 3.0은 반드시 trust_remote_code=True 설정을 넣어 모델 전용 아키텍처 코드를 빌드해야 합니다)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    device_map=device_map,
    low_cpu_mem_usage=low_cpu_mem_usage,
    trust_remote_code=True
)

print("Model & Tokenizer loaded successfully!")